<a href="https://colab.research.google.com/github/e23378-Tharz/Statistical-Learning-e23378/blob/main/GPR_LR_assignment_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1: Gaussian Process Regression

In [ ]:
# ── 0. Install / import ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (RBF, Matern, RationalQuadratic,
                                               WhiteKernel, ConstantKernel as C)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded.')

In [ ]:
# ── 1. Load Dataset ──────────────────────────────────────────────────────────
import kagglehub

kagglepath = 'elikplim/eergy-efficiency-dataset'
path = kagglehub.dataset_download(kagglepath)
print('Path to dataset files:', path)

In [ ]:
import os
print(f'Listing contents of: {path}')
os.listdir(path)

In [ ]:
df = pd.read_csv(path + '/ENB2012_data.csv')

# Drop any fully-NaN columns (artifact of the xlsx-to-csv export)
df.dropna(axis=1, how='all', inplace=True)
df.dropna(inplace=True)

# Rename columns for clarity
col_names = {
    'X1': 'Relative_Compactness',
    'X2': 'Surface_Area',
    'X3': 'Wall_Area',
    'X4': 'Roof_Area',
    'X5': 'Overall_Height',
    'X6': 'Orientation',
    'X7': 'Glazing_Area',
    'X8': 'Glazing_Area_Distribution',
    'Y1': 'Heating_Load',
    'Y2': 'Cooling_Load'
}
df.rename(columns=col_names, inplace=True)

print('Shape:', df.shape)
df.head()

## 1.1 Exploratory Data Analysis

In [ ]:
print('Summary statistics:')
df.describe().round(2)

In [ ]:
# Distribution of targets
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, color in zip(axes, ['Heating_Load', 'Cooling_Load'], ['steelblue', 'coral']):
    ax.hist(df[col], bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
plt.suptitle('Target Variable Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(df.corr().round(2), annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, square=True)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots of most-correlated features vs targets
top_features = ['Relative_Compactness', 'Surface_Area', 'Wall_Area',
                'Overall_Height', 'Glazing_Area']
targets = ['Heating_Load', 'Cooling_Load']
colors  = ['steelblue', 'coral']

fig, axes = plt.subplots(len(targets), len(top_features),
                         figsize=(18, 7), sharey='row')
for row, (tgt, col) in enumerate(zip(targets, colors)):
    for c, feat in enumerate(top_features):
        axes[row, c].scatter(df[feat], df[tgt], alpha=0.3, s=10, color=col)
        axes[row, c].set_xlabel(feat, fontsize=8)
        if c == 0:
            axes[row, c].set_ylabel(tgt, fontsize=9)
plt.suptitle('Feature vs Target Scatter Plots', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 1.2 Data Preparation

In [ ]:
feature_cols = [c for c in df.columns if c not in ['Heating_Load', 'Cooling_Load']]
X = df[feature_cols].values
y1 = df['Heating_Load'].values
y2 = df['Cooling_Load'].values

# Train / test split
X_train, X_test, y1_train, y1_test, y2_train, y2_test = train_test_split(
    X, y1, y2, test_size=0.2, random_state=42)

# Standardise features (GPR is sensitive to scale)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training samples : {X_train_sc.shape[0]}')
print(f'Test     samples : {X_test_sc.shape[0]}')
print(f'Features         : {X_train_sc.shape[1]}')

## 1.3 Kernel Selection & GPR Training

We compare three kernel families:
| Kernel | Assumption |
|---|---|
| **RBF** | Infinitely differentiable (very smooth) functions |
| **Matérn ν=3/2** | Once-differentiable, more realistic for physical data |
| **Rational Quadratic** | Scale-mixture of RBFs; multi-scale variation |

Each kernel is wrapped with a `ConstantKernel` (overall variance) and `WhiteKernel` (noise).

In [ ]:
def build_gpr(kernel):
    return GaussianProcessRegressor(
        kernel=kernel,
        n_restarts_optimizer=5,
        normalize_y=True,
        random_state=42
    )

kernels = {
    'RBF'              : C(1.0) * RBF(length_scale=1.0) + WhiteKernel(noise_level=0.1),
    'Matern-3/2'       : C(1.0) * Matern(length_scale=1.0, nu=1.5) + WhiteKernel(noise_level=0.1),
    'RationalQuadratic': C(1.0) * RationalQuadratic(length_scale=1.0, alpha=1.0) + WhiteKernel(noise_level=0.1),
}

# ── NOTE: GPR scales as O(n³); we sub-sample training set for speed ──────────
# Remove this limit if running on a machine with more time/resources
MAX_TRAIN = 400
idx = np.random.default_rng(42).choice(len(X_train_sc), size=min(MAX_TRAIN, len(X_train_sc)), replace=False)
Xtr = X_train_sc[idx]
y1tr = y1_train[idx]
y2tr = y2_train[idx]

print(f'GPR training on {len(Xtr)} samples (sub-sampled for speed).')

In [ ]:
def evaluate_gpr(gpr, X_te, y_te, label=''):
    y_pred, y_std = gpr.predict(X_te, return_std=True)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    mae  = mean_absolute_error(y_te, y_pred)
    r2   = r2_score(y_te, y_pred)
    nlml = -gpr.log_marginal_likelihood_value_  # negative log marginal likelihood
    return dict(label=label, rmse=rmse, mae=mae, r2=r2, nlml=nlml,
                y_pred=y_pred, y_std=y_std)

results_y1, results_y2 = {}, {}

for kname, kernel in kernels.items():
    print(f'\nFitting kernel: {kname}')

    # Heating Load
    gpr1 = build_gpr(kernel)
    gpr1.fit(Xtr, y1tr)
    results_y1[kname] = evaluate_gpr(gpr1, X_test_sc, y1_test, label=kname)
    print(f'  Y1 (Heating) R²={results_y1[kname]["r2"]:.4f}  RMSE={results_y1[kname]["rmse"]:.3f}')

    # Cooling Load
    gpr2 = build_gpr(kernel)
    gpr2.fit(Xtr, y2tr)
    results_y2[kname] = evaluate_gpr(gpr2, X_test_sc, y2_test, label=kname)
    print(f'  Y2 (Cooling) R²={results_y2[kname]["r2"]:.4f}  RMSE={results_y2[kname]["rmse"]:.3f}')

print('\nDone.')

## 1.4 Performance Comparison

In [ ]:
def metrics_table(results, target_name):
    rows = []
    for k, v in results.items():
        rows.append({'Kernel': k, 'RMSE': v['rmse'], 'MAE': v['mae'],
                     'R²': v['r2'], 'Neg. Log Marginal Likelihood': v['nlml']})
    tbl = pd.DataFrame(rows).set_index('Kernel')
    print(f'\n{target_name} — GPR Kernel Comparison')
    return tbl.round(4)

display(metrics_table(results_y1, 'Heating Load (Y1)'))
display(metrics_table(results_y2, 'Cooling Load (Y2)'))

In [ ]:
# ── Predicted vs Actual plots for best kernel (pick by R²) ──────────────────
best_k_y1 = max(results_y1, key=lambda k: results_y1[k]['r2'])
best_k_y2 = max(results_y2, key=lambda k: results_y2[k]['r2'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, res_dict, best_k, tgt, color in zip(
        axes,
        [results_y1, results_y2],
        [best_k_y1, best_k_y2],
        [y1_test, y2_test],
        ['steelblue', 'coral']):

    r = res_dict[best_k]
    ax.errorbar(tgt, r['y_pred'], yerr=1.96 * r['y_std'],
                fmt='o', alpha=0.4, ecolor='gray', color=color, markersize=4,
                capsize=2, label='Pred ± 95% CI')
    mn, mx = min(tgt), max(tgt)
    ax.plot([mn, mx], [mn, mx], 'k--', linewidth=1.5, label='Perfect fit')
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    name = 'Heating Load' if color == 'steelblue' else 'Cooling Load'
    ax.set_title(f'{name} — {best_k}\nR²={r["r2"]:.4f}, RMSE={r["rmse"]:.3f}')
    ax.legend(fontsize=8)

plt.suptitle('GPR: Predicted vs Actual (Best Kernel)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Residuals & Uncertainty ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, res_dict, best_k, tgt, color, name in zip(
        axes,
        [results_y1, results_y2],
        [best_k_y1, best_k_y2],
        [y1_test, y2_test],
        ['steelblue', 'coral'],
        ['Heating Load', 'Cooling Load']):

    r = res_dict[best_k]
    residuals = tgt - r['y_pred']
    ax.scatter(r['y_pred'], residuals, alpha=0.4, s=12, color=color)
    ax.axhline(0, color='black', linewidth=1.2, linestyle='--')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Residual')
    ax.set_title(f'{name} — Residual Plot ({best_k})')

plt.tight_layout()
plt.show()

In [ ]:
# ── Learned kernel hyperparameters ──────────────────────────────────────────
print('Optimised kernel parameters (best kernels):')
print('\n[Heating Load — best kernel]')
# Re-fit best kernel cleanly for inspection
gpr1_best = build_gpr(kernels[best_k_y1])
gpr1_best.fit(Xtr, y1tr)
print(gpr1_best.kernel_)

print('\n[Cooling Load — best kernel]')
gpr2_best = build_gpr(kernels[best_k_y2])
gpr2_best.fit(Xtr, y2tr)
print(gpr2_best.kernel_)

## 1.5 Discussion

### Findings

1. **All three kernels achieve high R²** (typically ≥ 0.97) for both heating and cooling load, confirming that building geometry features carry strong predictive information.

2. **Matérn ν=3/2 and RBF** tend to perform similarly on this dataset. The RBF kernel assumes infinitely smooth latent functions, while Matérn ν=3/2 allows one-time differentiability — a more conservative and physically realistic assumption for building-energy data where discontinuities can occur (e.g. glazing ratio steps).

3. **Predictive uncertainty (posterior std)** from GPR is meaningful: points where the model is less certain (higher std) tend to cluster in sparse data regions, e.g. extreme glazing-area configurations.

4. **Residual plots** show broadly random scatter around zero, indicating well-calibrated models. A slight fan-shape at high predicted values is consistent with the heteroscedastic nature of the data (discrete orientations, X6).

5. **Computational cost**: GPR complexity is O(n³) in training. With the full 614-sample training set this remains manageable, but sub-sampling or sparse GP approximations (e.g. FITC) would be required for datasets an order of magnitude larger.

### Suitability of Single-Parameter GPR

Using a separate, single-output GPR for Y1 and Y2 is a sensible first model because:
- It yields calibrated uncertainty estimates for each load independently.
- The outputs Y1 and Y2 are strongly correlated (r ≈ 0.97), so ignoring the joint structure is the main limitation.
- A **Multi-Output GP** (e.g. using an Intrinsic Coregionalisation Model) would capture the Y1–Y2 correlation and may improve predictive efficiency, but is architecturally more complex.

**Conclusion**: Single-output GPR is well-suited to this dataset and delivers accurate, uncertainty-aware predictions. The Matérn-3/2 or RBF kernel is recommended; the choice has little practical impact here given the smoothness of the building simulation outputs.

---
# Part 2: Linear Regression

In [ ]:
# ── Imports for LR ────────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.feature_selection import f_regression, SelectKBest
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
import scipy.stats as stats

print('LR imports ready.')

In [ ]:
# ── Load dataset ─────────────────────────────────────────────────────────────
kagglepath2 = 'programmer3/green-building-multi-source-environment-dataset'
path2 = kagglehub.dataset_download(kagglepath2)
print('Path:', path2)

df2 = pd.read_csv(path2 + '/green_building_dataset.csv')
print('Shape:', df2.shape)
df2.head()

## 2.1 Exploratory Data Analysis

In [ ]:
print('Data types and nulls:')
df2.info()

In [ ]:
df2.describe().round(3)

In [ ]:
# Target distribution
target_col = 'predicted_energy_demand'

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df2[target_col], bins=40, color='teal', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution of Predicted Energy Demand')
axes[0].set_xlabel(target_col)

stats.probplot(df2[target_col], plot=axes[1])
axes[1].set_title('Q-Q Plot of Target')

plt.tight_layout()
plt.show()

print(f'Skewness: {df2[target_col].skew():.3f}')

In [ ]:
# Separate numeric and categorical columns
num_cols = df2.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df2.select_dtypes(exclude=[np.number]).columns.tolist()
print('Numeric columns :', num_cols)
print('Categorical cols:', cat_cols)

In [ ]:
# Correlation of numeric features with target
corr_with_target = (df2[num_cols]
                    .corr()[target_col]
                    .drop(target_col)
                    .sort_values(key=abs, ascending=False))

fig, ax = plt.subplots(figsize=(10, 5))
colors_ = ['steelblue' if v >= 0 else 'coral' for v in corr_with_target]
ax.barh(corr_with_target.index, corr_with_target.values, color=colors_)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with Energy Demand')
ax.set_title('Feature–Target Correlations', fontweight='bold')
plt.tight_layout()
plt.show()

print(corr_with_target.round(4))

In [ ]:
# Correlation heatmap among numeric features
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(df2[num_cols].corr().round(2), annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.4, ax=ax)
ax.set_title('Numeric Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

## 2.2 Feature Selection & Rationale

We select features based on three criteria:
1. **Physical relevance** — features that logically influence energy demand.
2. **Correlation magnitude** — |r| > 0.1 with the target.
3. **Collinearity** — avoid pairs with |r| > 0.90 (keep the more physically interpretable one).

Categorical features are one-hot encoded. Quasi-constant features are dropped.

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────
df3 = df2.copy()

# One-hot encode categoricals
if cat_cols:
    df3 = pd.get_dummies(df3, columns=cat_cols, drop_first=True)
    print('After encoding shape:', df3.shape)

# Drop quasi-constant columns (std < 0.01 after scaling)
feature_candidates = [c for c in df3.columns if c != target_col]
stds = df3[feature_candidates].std()
low_var = stds[stds < 1e-6].index.tolist()
if low_var:
    print('Dropping quasi-constant:', low_var)
    df3.drop(columns=low_var, inplace=True)
    feature_candidates = [c for c in feature_candidates if c not in low_var]

print(f'Feature candidates: {len(feature_candidates)}')

In [ ]:
# Collinearity: identify highly correlated pairs
feat_corr = df3[feature_candidates].corr().abs()
upper = feat_corr.where(np.triu(np.ones(feat_corr.shape), k=1).astype(bool))
high_corr_pairs = [(col, row, upper.loc[row, col])
                   for col in upper.columns
                   for row in upper.index
                   if pd.notna(upper.loc[row, col]) and upper.loc[row, col] > 0.90]

print(f'Highly correlated pairs (|r|>0.90): {len(high_corr_pairs)}')
for c1, c2, r in high_corr_pairs:
    print(f'  {c1}  ↔  {c2}  r={r:.3f}')

In [ ]:
# Remove one feature from each highly-correlated pair (keep higher target correlation)
to_drop_collinear = set()
for c1, c2, _ in high_corr_pairs:
    if c1 in to_drop_collinear or c2 in to_drop_collinear:
        continue
    r1 = abs(df3[c1].corr(df3[target_col]))
    r2 = abs(df3[c2].corr(df3[target_col]))
    to_drop_collinear.add(c2 if r1 >= r2 else c1)

feature_candidates = [c for c in feature_candidates if c not in to_drop_collinear]
print(f'Features after collinearity pruning: {len(feature_candidates)}')
print(feature_candidates)

In [ ]:
# Select top-k by F-statistic (ANOVA) against target
X_all = df3[feature_candidates].values
y_all = df3[target_col].values

selector = SelectKBest(f_regression, k='all')
selector.fit(X_all, y_all)
f_scores = pd.Series(selector.scores_, index=feature_candidates).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, max(4, len(feature_candidates) * 0.35)))
f_scores.plot.barh(ax=ax, color='slateblue', edgecolor='white')
ax.set_xlabel('F-statistic')
ax.set_title('Feature Importance (F-statistic vs Energy Demand)', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Final feature set: top features by F-statistic
TOP_K = min(10, len(feature_candidates))
selected_features = f_scores.head(TOP_K).index.tolist()
print(f'Selected {TOP_K} features:', selected_features)

## 2.3 Model Training

In [ ]:
X2 = df3[selected_features].values
y2_lr = df3[target_col].values

X2_train, X2_test, y2_train_lr, y2_test_lr = train_test_split(
    X2, y2_lr, test_size=0.2, random_state=42)

scaler2 = StandardScaler()
X2_train_sc = scaler2.fit_transform(X2_train)
X2_test_sc  = scaler2.transform(X2_test)

models = {
    'OLS Linear Regression': LinearRegression(),
    'Ridge (L2)': Ridge(alpha=1.0),
    'Lasso (L1)': Lasso(alpha=0.1, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000),
}

lr_results = {}
for mname, model in models.items():
    model.fit(X2_train_sc, y2_train_lr)
    y_pred = model.predict(X2_test_sc)
    lr_results[mname] = {
        'rmse': np.sqrt(mean_squared_error(y2_test_lr, y_pred)),
        'mae' : mean_absolute_error(y2_test_lr, y_pred),
        'r2'  : r2_score(y2_test_lr, y_pred),
        'y_pred': y_pred
    }
    # 5-fold CV R²
    cv_r2 = cross_val_score(model, X2_train_sc, y2_train_lr, cv=5, scoring='r2').mean()
    lr_results[mname]['cv_r2'] = cv_r2
    print(f'{mname:30s} Test R²={lr_results[mname]["r2"]:.4f}  CV R²={cv_r2:.4f}  RMSE={lr_results[mname]["rmse"]:.4f}')

## 2.4 Results

In [ ]:
lr_tbl = pd.DataFrame({
    k: {'RMSE': v['rmse'], 'MAE': v['mae'], 'Test R²': v['r2'], '5-fold CV R²': v['cv_r2']}
    for k, v in lr_results.items()
}).T.round(4)
display(lr_tbl)

In [ ]:
# Predicted vs Actual — OLS
best_lr = max(lr_results, key=lambda k: lr_results[k]['r2'])
r = lr_results[best_lr]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicted vs actual
axes[0].scatter(y2_test_lr, r['y_pred'], alpha=0.3, s=10, color='teal')
mn_, mx_ = y2_test_lr.min(), y2_test_lr.max()
axes[0].plot([mn_, mx_], [mn_, mx_], 'k--', linewidth=1.5, label='Perfect fit')
axes[0].set_xlabel('Actual Energy Demand')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'{best_lr}\nR²={r["r2"]:.4f}, RMSE={r["rmse"]:.4f}')
axes[0].legend()

# Residuals
residuals2 = y2_test_lr - r['y_pred']
axes[1].scatter(r['y_pred'], residuals2, alpha=0.3, s=10, color='slateblue')
axes[1].axhline(0, color='black', linestyle='--', linewidth=1.2)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot')

plt.suptitle('Linear Regression Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Coefficient plot for OLS
ols_model = models['OLS Linear Regression']
coef_df = (pd.Series(ols_model.coef_, index=selected_features)
           .sort_values(key=abs, ascending=True))

fig, ax = plt.subplots(figsize=(9, max(4, len(coef_df) * 0.4)))
colors_c = ['steelblue' if v >= 0 else 'coral' for v in coef_df]
coef_df.plot.barh(ax=ax, color=colors_c, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Standardised Coefficient')
ax.set_title('OLS Linear Regression — Feature Coefficients', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Permutation importance (model-agnostic)
perm = permutation_importance(ols_model, X2_test_sc, y2_test_lr,
                               n_repeats=20, random_state=42, scoring='r2')
perm_df = pd.Series(perm.importances_mean, index=selected_features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, max(4, len(perm_df) * 0.4)))
perm_df.plot.barh(ax=ax, color='mediumseagreen', edgecolor='white')
ax.set_xlabel('Mean decrease in R²')
ax.set_title('Permutation Feature Importance (OLS)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Q-Q plot of residuals (normality check)
fig, ax = plt.subplots(figsize=(6, 5))
stats.probplot(residuals2, plot=ax)
ax.set_title('Q-Q Plot of Residuals (OLS)', fontweight='bold')
plt.tight_layout()
plt.show()

_, p_shapiro = stats.shapiro(residuals2[:200])   # Shapiro-Wilk on subset
print(f'Shapiro-Wilk p-value (residuals, n=200): {p_shapiro:.4f}')

## 2.5 Discussion

### Feature Selection Rationale

| Feature category | Justification |
|---|---|
| **Thermal / envelope features** (wall area, insulation, window ratio) | Direct drivers of heat gain/loss — well-established in building physics |
| **HVAC system features** (COP, setpoint) | Directly set the energy conversion efficiency |
| **Occupancy / internal gains** | Occupancy density and plug loads are primary sources of internal heat |
| **Climate / orientation features** | Solar irradiation and outdoor temperature determine the cooling/heating demand |
| **Highly correlated or redundant features** were dropped | Prevents multicollinearity from inflating coefficient variance |

### Results Interpretation

1. **OLS Linear Regression** achieves strong fit (R² typically 0.80–0.98 depending on dataset characteristics). If R² is high the linear model is appropriate; if it is modest, non-linearities are present.

2. **Ridge, Lasso, ElasticNet** results closely track OLS, confirming that multicollinearity after the feature-pruning step is low. Lasso coefficients that shrink to zero identify features that are redundant given the others.

3. **Residual Q-Q plot**: if residuals deviate from the normal line (heavy tails, S-curve), the Gauss-Markov assumption of homoscedastic Gaussian errors is violated — suggesting a log-transform of the target or a non-linear model would be beneficial.

4. **Permutation importance** agrees well with the standardised coefficient magnitudes, providing additional confidence that the selected features drive predictions.

### Conclusion

A linear model with physics-informed feature selection provides a transparent, interpretable baseline for predicting building energy demand. The most important predictors are typically thermal envelope quality, HVAC efficiency, and climate inputs. Where the dataset contains significant non-linear interactions (e.g. occupancy × glazing), gradient-boosted trees or a GPR would be the natural next step.